# EEG 运动想象实验总流程

该 Notebook 为**模块化实验台**,支持:

- 单被试完整流程
- 多特征方法对比
- 10折交叉验证
- 同被试跨会话验证 (A01T → A01E)
- 跨被试泛化验证 (如 A01T → A03E)
- 批量泛化结果汇总

推荐逐节运行,而不是一次性全部运行。

## 0. 环境与路径准备

In [4]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

project_root = Path.cwd().parent
code_dir = project_root / 'code'
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
if str(code_dir) not in sys.path:
    sys.path.insert(0, str(code_dir))

# 清理可能存在的 code 模块缓存
if 'code' in sys.modules:
    del sys.modules['code']

import code._plot_config

from code.config import DEFAULT_CONFIG, TASK_CLASS_NAMES, epochs_events_to_class_labels
from pretreatment.complete_preprocessing import complete_preprocessing_pipeline
from classification.svm_classifier import train_eeg_svm_pipeline, plot_confusion_matrix
from test_on_evaluation_set import (
    preprocess_evaluation_set,
    load_true_labels,
    align_labels_with_epochs,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import accuracy_score, confusion_matrix, cohen_kappa_score

print(f'项目根目录: {project_root}')
print(f'代码目录: {code_dir}')
print('✅ 环境准备完成')

项目根目录: f:\Graduation Design
代码目录: f:\Graduation Design\code
✅ 环境准备完成


## 1. 实验配置

你可以只改这里,不需要改后面的代码。

In [ ]:
cfg = DEFAULT_CONFIG

TRAIN_SUBJECT = 'A01T'
EVAL_SUBJECT = 'A01E'
CROSS_SUBJECT_EVAL = 'A03E'

FEATURE_CONFIGS = [
    ('csp', 'CSP', dict()),
    ('wavelet', 'Wavelet-All', dict(motor_channels_only=False)),
    ('fused', 'Fused-All', dict(motor_channels_only=False)),
    ('fused', 'Fused-Motor', dict(motor_channels_only=True)),
    ('fb_csp', 'FBCSP', dict(freq_bands=[(8,12),(12,16),(16,20),(20,24),(24,30)])),
]

print('当前训练集:', TRAIN_SUBJECT)
print('当前同被试验证集:', EVAL_SUBJECT)
print('当前跨被试验证集:', CROSS_SUBJECT_EVAL)
print('方法数:', len(FEATURE_CONFIGS))

## 2. 训练集预处理 (T 会话)

In [ ]:
epochs_train, ica_train = complete_preprocessing_pipeline(subject=TRAIN_SUBJECT)
y_train = epochs_events_to_class_labels(epochs_train)

print('✅ 训练集预处理完成')
print('Epochs shape:', epochs_train.get_data().shape)
print('标签分布:', dict(zip(*np.unique(y_train, return_counts=True))))

## 3. 单方法独立运行:选择一个特征方法

如果你只想测试某一种方法,把 `METHOD_INDEX` 改掉即可。

In [ ]:
METHOD_INDEX = 4
feature_set, display_name, extra_kwargs = FEATURE_CONFIGS[METHOD_INDEX]

clf_single, cv_scores_single, acc_single = train_eeg_svm_pipeline(
    epochs_train,
    y_train,
    feature_set=feature_set,
    cv_folds=cfg.cv_folds,
    n_csp_components=cfg.csp_components,
    wavelet=cfg.wavelet,
    wavelet_level=cfg.wavelet_level,
    kernel=cfg.svm_kernel,
    random_state=cfg.random_state,
    **extra_kwargs,
)

print(f'✅ {display_name} 完成')
print(f'准确率: {acc_single:.4f} ± {cv_scores_single.std():.4f}')

## 4. 全部方法 10 折交叉验证对比

In [ ]:
results = {}
classifiers = {}

for feature_set, display_name, extra_kwargs in FEATURE_CONFIGS:
    print('\n' + '='*70)
    print('运行方法:', display_name)
    clf, cv_scores, acc = train_eeg_svm_pipeline(
        epochs_train,
        y_train,
        feature_set=feature_set,
        cv_folds=cfg.cv_folds,
        n_csp_components=cfg.csp_components,
        wavelet=cfg.wavelet,
        wavelet_level=cfg.wavelet_level,
        kernel=cfg.svm_kernel,
        random_state=cfg.random_state,
        **extra_kwargs,
    )
    results[display_name] = {
        'feature_set': feature_set,
        'cv_scores': cv_scores,
        'cv_mean': acc,
        'cv_std': cv_scores.std(),
        'extra_kwargs': extra_kwargs,
    }
    classifiers[display_name] = clf

summary_df = pd.DataFrame([
    {
        'method': name,
        'cv_mean': res['cv_mean'],
        'cv_std': res['cv_std'],
    }
    for name, res in results.items()
]).sort_values('cv_mean', ascending=False).reset_index(drop=True)

summary_df

## 5. 交叉验证混淆矩阵 (最佳方法)

In [ ]:
best_method = summary_df.iloc[0]['method']
best_clf = classifiers[best_method]

X_train = epochs_train.get_data()
cv = StratifiedKFold(n_splits=cfg.cv_folds, shuffle=True, random_state=cfg.random_state)
y_pred_cv = cross_val_predict(best_clf, X_train, y_train, cv=cv)
cm_cv = confusion_matrix(y_train, y_pred_cv, labels=[1,2,3,4])

plot_confusion_matrix(cm_cv, class_names=TASK_CLASS_NAMES, save_path=f'./{TRAIN_SUBJECT}_{best_method}_cv_confusion_matrix.png')
print('最佳方法:', best_method)
print('✅ 交叉验证混淆矩阵已保存')

## 6. 同被试跨会话验证 (T → E)

这里验证:例如 **A01T 训练 → A01E 测试**。

In [ ]:
epochs_eval_same, ica_eval_same = preprocess_evaluation_set(EVAL_SUBJECT, config=cfg)
y_eval_same_all = load_true_labels(EVAL_SUBJECT)
y_eval_same = align_labels_with_epochs(epochs_eval_same, y_eval_same_all)

same_subject_eval_rows = []
for method_name, clf in classifiers.items():
    y_pred = clf.predict(epochs_eval_same.get_data())
    acc = accuracy_score(y_eval_same, y_pred)
    kappa = cohen_kappa_score(y_eval_same, y_pred)
    same_subject_eval_rows.append({
        'method': method_name,
        'accuracy': acc,
        'kappa': kappa,
    })

same_subject_eval_df = pd.DataFrame(same_subject_eval_rows).sort_values('accuracy', ascending=False).reset_index(drop=True)
same_subject_eval_df

## 7. 单次跨被试泛化验证 (Train Subject T → Other Subject E)

这里验证:例如 **A01T 训练 → A03E 测试**。

如果这个结果也不错,说明模型学到了一定的通用模式。

In [ ]:
epochs_eval_cross, ica_eval_cross = preprocess_evaluation_set(CROSS_SUBJECT_EVAL, config=cfg)
y_eval_cross_all = load_true_labels(CROSS_SUBJECT_EVAL)
y_eval_cross = align_labels_with_epochs(epochs_eval_cross, y_eval_cross_all)

cross_subject_eval_rows = []
for method_name, clf in classifiers.items():
    y_pred = clf.predict(epochs_eval_cross.get_data())
    acc = accuracy_score(y_eval_cross, y_pred)
    kappa = cohen_kappa_score(y_eval_cross, y_pred)
    cross_subject_eval_rows.append({
        'method': method_name,
        'accuracy': acc,
        'kappa': kappa,
    })

cross_subject_eval_df = pd.DataFrame(cross_subject_eval_rows).sort_values('accuracy', ascending=False).reset_index(drop=True)
cross_subject_eval_df

## 8. 批量跨被试泛化实验

固定一个训练被试 T,会在多个其他被试 E 上测试。

例如:训练 A01T,然后测试 A02E ~ A09E。

In [ ]:
RUN_BATCH_CROSS_SUBJECT = False

if RUN_BATCH_CROSS_SUBJECT:
    train_prefix = TRAIN_SUBJECT[:3]
    target_eval_subjects = [f'A{i:02d}E' for i in range(1, 10) if f'A{i:02d}' != train_prefix]
    batch_rows = []

    for eval_subj in target_eval_subjects:
        print('\n' + '='*80)
        print('跨被试测试目标:', eval_subj)
        epochs_eval_tmp, _ = preprocess_evaluation_set(eval_subj, config=cfg)
        y_eval_tmp = align_labels_with_epochs(epochs_eval_tmp, load_true_labels(eval_subj))

        for method_name, clf in classifiers.items():
            y_pred_tmp = clf.predict(epochs_eval_tmp.get_data())
            batch_rows.append({
                'train_subject': TRAIN_SUBJECT,
                'eval_subject': eval_subj,
                'method': method_name,
                'accuracy': accuracy_score(y_eval_tmp, y_pred_tmp),
                'kappa': cohen_kappa_score(y_eval_tmp, y_pred_tmp),
            })

    batch_cross_subject_df = pd.DataFrame(batch_rows)
    display(batch_cross_subject_df.sort_values(['method', 'accuracy'], ascending=[True, False]))
else:
    print('当前未运行批量跨被试实验。将 RUN_BATCH_CROSS_SUBJECT 改为 True 后再运行本单元。')

## 9. 结果汇总与建议

你最终可以回答这些问题:

1. 单被试 10 折 CV 下哪种方法最好?
2. 同被试跨会话 (T→E) 是否稳定?
3. 跨被试 (A01T→A03E) 是否仍然有效?
4. 多个其他被试 E 上平均表现如何?